# Regional consumption report 2023

Goal: for each region, how much do our customers actually consume versus their annual estimate,
and how does it split by tariff. The commercial team will use this to set regional price changes.

Data: `meters.csv` (customer master), `meter_readings_daily.csv` (daily kWh per meter, 2023).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## Load data

In [2]:
meters = pd.read_csv("../data/meters.csv")
readings = pd.read_csv("../data/meter_readings_daily.csv")
print(meters.shape, readings.shape)
meters.head()

(300, 7) (107503, 3)


,meter_id,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,London,Fixed,sme,21622.0,2021-07-07,False
1,M100001,London,Fixed,residential,2286.0,2022-11-17,False
2,M100002,London,Fixed,residential,3665.0,2021-06-04,False
3,M100003,Scotland,Fixed,residential,2575.0,2021-10-26,False
4,M100004,Midlands,Fixed,residential,2191.0,2022-10-31,False


In [3]:
readings.head()

,meter_id,date,kwh
0,M100000,2023-01-01,96.882
1,M100000,2023-01-02,111.294
2,M100000,2023-01-03,64.984
3,M100000,2023-01-04,178.334
4,M100000,2023-01-05,107.423


In [4]:
print(meters["region"].unique())
meters.isna().sum()

['London' 'Scotland' 'Midlands' 'North' 'Wales' 'london' 'wales' 'north'
 'midlands']


meter_id                0
region                  0
tariff                 13
customer_type           0
annual_kwh_estimate     6
signup_date             0
has_solar               0
dtype: int64

## Clean the meter master

In [5]:
# a missing tariff means the customer is on the default Fixed tariff
meters["tariff"] = meters["tariff"].fillna("Fixed")
meters["annual_kwh_estimate"] = meters["annual_kwh_estimate"].fillna(0)
meters["signup_date"] = pd.to_datetime(meters["signup_date"])
meters["tariff"].value_counts()

tariff
Fixed       156
Variable     94
TOU          50
Name: count, dtype: int64

## Clean the readings

In [6]:
readings["date"] = pd.to_datetime(readings["date"])

# readings above 150 kWh/day are meter faults
readings.loc[readings["kwh"] > 150, "kwh"] = np.nan

readings["kwh"].describe().round(1)

count    107246.0
mean         15.4
std          21.0
min           0.5
25%           6.2
50%           8.8
75%          12.9
max         149.9
Name: kwh, dtype: float64

## Join readings to the meter master

In [7]:
merged = readings.merge(meters, on="meter_id", how="inner")
print(merged.shape)
merged.head()

(107303, 9)


,meter_id,date,kwh,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,2023-01-01,96.882,London,Fixed,sme,21622.0,2021-07-07,False
1,M100000,2023-01-02,111.294,London,Fixed,sme,21622.0,2021-07-07,False
2,M100000,2023-01-03,64.984,London,Fixed,sme,21622.0,2021-07-07,False
3,M100000,2023-01-04,NaN,London,Fixed,sme,21622.0,2021-07-07,False
4,M100000,2023-01-05,107.423,London,Fixed,sme,21622.0,2021-07-07,False


## Tariff unit rates

In [8]:
rates = pd.DataFrame({
    "tariff":      ["Fixed", "Variable", "TOU",  "TOU"],
    "period":      ["all",   "all",      "peak", "offpeak"],
    "unit_rate_p": [24.5,    27.1,       35.0,   12.0],
})
merged = merged.merge(rates, on="tariff")
merged["cost_gbp"] = merged["kwh"] * merged["unit_rate_p"] / 100
print(merged.shape)
merged[["meter_id", "date", "kwh", "tariff", "period", "cost_gbp"]].head()

(125191, 12)


,meter_id,date,kwh,tariff,period,cost_gbp
0,M100000,2023-01-01,96.882,Fixed,all,23.736090
1,M100000,2023-01-02,111.294,Fixed,all,27.267030
2,M100000,2023-01-03,64.984,Fixed,all,15.921080
3,M100000,2023-01-04,NaN,Fixed,all,NaN
4,M100000,2023-01-05,107.423,Fixed,all,26.318635


## Per-meter totals

In [9]:
per_meter = (
    merged.groupby(["meter_id", "region", "tariff", "customer_type"], as_index=False)
          .agg(total_kwh=("kwh", "sum"),
               n_readings=("kwh", "size"),
               total_cost=("cost_gbp", "sum"))
)
per_meter = per_meter.merge(meters[["meter_id", "annual_kwh_estimate"]], on="meter_id")
per_meter["actual_vs_estimate"] = per_meter["total_kwh"] / (per_meter["annual_kwh_estimate"] + 1)
per_meter.describe().round(1)

,total_kwh,n_readings,total_cost,annual_kwh_estimate,actual_vs_estimate
count,300.0,300.0,300.0,300.0,300.0
mean,6324.6,417.3,1579.2,5525.6,76.5
std,8098.9,133.6,1994.2,7276.4,555.8
min,938.3,350.0,229.9,0.0,0.8
25%,2826.3,356.0,718.0,2642.0,1.0
50%,3562.4,358.0,900.4,3271.5,1.0
75%,5202.4,361.0,1259.3,3983.0,1.0
max,59313.2,724.0,13938.6,36818.0,6507.5


## Regional summary

In [10]:
summary = per_meter.groupby("region").agg(
    meters=("meter_id", "nunique"),
    readings=("n_readings", "sum"),
    avg_kwh=("total_kwh", "mean"),
    avg_cost=("total_cost", "mean"),
    actual_vs_estimate=("actual_vs_estimate", "mean"),
)
summary.round(2)

,meters,readings,avg_kwh,avg_cost,actual_vs_estimate
region,,,,,
London,96,40042,6904.44,1708.07,34.68
Midlands,44,18987,7827.17,1943.22,148.89
North,65,27560,5973.66,1513.01,101.28
Scotland,57,23223,4549.83,1149.27,56.48
Wales,32,13239,6877.81,1711.20,101.56
london,3,1077,3559.22,903.71,1.02
midlands,1,355,3135.62,849.75,0.97
north,1,351,3853.49,944.10,0.97
wales,1,357,4762.37,1166.78,1.01


## Total kWh by region and tariff

In [11]:
merged.pivot_table(index="region", columns="tariff", values="kwh").round(1)

tariff,Fixed,TOU,Variable
region,,,
London,17.6,17.9,13.4
Midlands,15.2,22.2,17.2
North,13.7,8.3,22.4
Scotland,10.6,8.0,15.4
Wales,24.5,9.7,11.6
london,9.8,NaN,10.1
midlands,NaN,NaN,8.8
north,11.0,NaN,NaN
wales,13.3,NaN,NaN


## Reduction targets from the commercial team

In [12]:
region_targets = pd.DataFrame({
    "region": ["LONDON", "MIDLANDS", "NORTH", "SCOTLAND", "WALES"],
    "target_reduction_pct": [5, 3, 3, 2, 3],
})
plan = summary.reset_index().merge(region_targets, on="region")
summary.head()

,meters,readings,avg_kwh,avg_cost,actual_vs_estimate
region,,,,,
London,96,40042,6904.444344,1708.072687,34.678643
Midlands,44,18987,7827.168773,1943.216329,148.891132
North,65,27560,5973.660415,1513.011325,101.276685
Scotland,57,23223,4549.831228,1149.266442,56.477499
Wales,32,13239,6877.814625,1711.195166,101.564488


## Results

In [13]:
ratio = summary.loc["London", "avg_kwh"] / summary.loc["Scotland", "avg_kwh"]
print(f"All {summary['meters'].sum()} meters covered, {summary['readings'].sum():,} readings.")
print(f"London customers use {ratio:.2f}x more electricity than Scottish customers.")
print(f"Customers consume {summary['actual_vs_estimate'].min():.0f}x to "
      f"{summary['actual_vs_estimate'].max():.0f}x their annual estimate - the estimates need rebuilding.")
print("Recommendation: apply the largest price increase to London, lowest to Scotland.")

All 300 meters covered, 125,191 readings.
London customers use 1.52x more electricity than Scottish customers.
Customers consume 1x to 149x their annual estimate - the estimates need rebuilding.
Recommendation: apply the largest price increase to London, lowest to Scotland.
